<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimisation_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
#!pip install qiskit # Uncomment installations upon first time running
#!pip install qiskit-aer

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT, QFTGate # Import QFTGate
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
import numpy as np
from math import gcd
from sympy.ntheory import factorint
import random
import time

#Qubit Reuse
#Instead of using separate qubits for each phase estimation bit, we can reuse a single qubit.

# Measures how long the  program takes to run
start_time = time.monotonic()

#Current Approach (uses n_count separate qubits)
#qc = QuantumCircuit(n_count + work_qubits, n_count)
#for i in range(n_count):
#    qc.h(i)
#    # ... controlled operations ...

#Optimized with Semi-classical QFT (single qubit reuse):
def semi_classical_shor(N, a):
    """
    Optimized Shor's algorithm using semi-classical QFT.
    Reuses a single counting qubit instead of n_count qubits.

    Optimization: Reduces qubit count from 2n+1 to n+2
    """
    n = N.bit_length()
    work_qubits = n

    # Only need 1 counting qubit + work qubits
    qc = QuantumCircuit(1 + work_qubits, 1)

    # Classical registers to store phase bits
    classical_bits = []

    # Perform iterative phase estimation
    for k in reversed(range(n*2)):  # From most significant to least
        # Initialize counting qubit to |+>
        qc.h(0)

        # Apply controlled-U^(2^k) for this bit
        # This is the modular exponentiation step
        apply_controlled_exponentiation(qc, a, 2**k, N,
                                        control_qubit=0,
                                        target_qubits=range(1, work_qubits+1))

        # Apply inverse QFT for this bit (semi-classical)
        # Phase correction based on previously measured bits
        for j, bit in enumerate(classical_bits):
            if bit == 1:
                qc.p(-np.pi / (2**(j+1)), 0)

        # Hadamard
        qc.h(0)

        # Measure
        qc.measure(0, 0)

        # Store result
        classical_bits.append(int(qc.measure(0, 0)))  # Simplified

        # Reset qubit for next iteration
        qc.reset(0)

    # Reconstruct phase from classical bits
    phase = sum(bit * 2**(-(i+1)) for i, bit in enumerate(classical_bits))

    # Find period using continued fractions
    r = Fraction(phase).limit_denominator(N).denominator

    return r

def apply_controlled_exponentiation(qc, a, exponent, N, control_qubit, target_qubits):
    """
    Apply controlled modular exponentiation U^(exponent)
    Optimized using repeated squaring and modular arithmetic
    """
    # For N=15, we can optimize further
    if N == 15:
        # Known patterns for modular multiplication
        # U|1> = |a>, U^2|1> = |a^2 mod 15>, etc.
        if a == 7:
            # Period 4 pattern
            if exponent % 4 == 1:
                # Apply mapping |1> → |7>
                qc.cx(control_qubit, target_qubits[0])
            elif exponent % 4 == 2:
                # Apply mapping |1> → |4>
                qc.cx(control_qubit, target_qubits[1])
            # ... etc.
    else:
        # Generic implementation using modular multiplication circuit
        pass

semi_classical_shor(15, 0)
print("Program took %s seconds to run" % (time.time() - start_time))

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'InstructionSet'